# Molecular Design VAE — Large CPU Mode

**~100k ZINC molecules · 400 epochs · CPU · ~4-6 hours**

**Important:** Free Colab will time out before training finishes.

This notebook shows results in cells — no live URL.

For a live URL use **Large GPU mode** (`colab_large_gpu.ipynb`).

## Cell 1 — Setup

In [ ]:
import os, subprocess

# Clone repo
if not os.path.exists('molecular-design-vae'):
    subprocess.run(['git', 'clone', 'https://github.com/Kaur-Simarpreet/molecular-design-vae.git'], check=True)

# Change directory — os.chdir works for all subsequent commands
os.chdir('molecular-design-vae')
print('Directory:', os.getcwd())

# Install dependencies
subprocess.run(['pip', 'install', '-q', 'torch', 'selfies',
                'flask', 'flask-cors', 'scipy', 'requests'])
subprocess.run(['pip', 'install', '-q', 'rdkit'])
print('Setup complete')


## Cell 2 — Train

Will likely time out on free Colab. Best checkpoint is saved as it trains.

In [ ]:
import os
if not os.path.exists('train_vae_extended.py'):
    os.chdir('molecular-design-vae')
print('Training from:', os.getcwd())
!python train_vae_extended.py --mode large


## Cell 3 — Generate molecules in notebook

Works even if training was interrupted — uses best checkpoint saved so far.

In [ ]:
import os, shutil, sys
if not os.path.exists('serve.py'):
    os.chdir('molecular-design-vae')

# Use best checkpoint if full training didn't complete
if os.path.exists('saved_model/vae_best.pt') and not os.path.exists('saved_model/vae.pt'):
    shutil.copy('saved_model/vae_best.pt', 'saved_model/vae.pt')
    print('Using best checkpoint')

sys.path.insert(0, '.')
from serve import vae, tokenizer, score_mol
import torch

weights = {'qed':0.30,'dock':0.40,'sa':0.15,'nov':0.05,'admet':0.10}
generated = []
for _ in range(30):
    z = torch.randn(1, vae.encoder.mu.out_features)
    smi = vae.decode_z(z, tokenizer, temperature=0.9)
    if smi:
        scored = score_mol(smi, 'EGFR kinase', weights=weights)
        if scored:
            generated.append(scored)
print(f'Generated {len(generated)} molecules')


## Cell 4 — Display results and export CSV

In [ ]:
import pandas as pd
df = pd.DataFrame(generated)
cols = ['smiles','qed','sa_score','docking_score','rl_reward','novelty']
available = [c for c in cols if c in df.columns]
print(df[available].to_string())
df.to_csv('large_results.csv', index=False)
print('Saved to large_results.csv')


## Cell 5 — Download results

In [ ]:
from google.colab import files
import shutil, os
if not os.path.exists('large_results.csv'):
    os.chdir('molecular-design-vae')
files.download('large_results.csv')
shutil.make_archive('/tmp/saved_model', 'zip', 'saved_model')
files.download('/tmp/saved_model.zip')
